# Install & Import Library

In [1]:
%pip install torch
%pip install transformers
%pip install conllu
%pip install pandas numpy scikit-learn matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import conllu
import os
import pandas as pd
import numpy as np
import re
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display
from collections import defaultdict

# Explore Dataset IndoLEM

## A. Explore Task Dataset - UD_Indonesian_GSD (RAW)

### 1. Set Path

In [3]:
BASE_PATH = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\raw\indolem-main\dependency_parsing"

PATH_GSD_TRAIN = os.path.join(BASE_PATH, "UD_Indonesian_GSD", "id_gsd-ud-train.conllu")
PATH_GSD_DEV   = os.path.join(BASE_PATH, "UD_Indonesian_GSD", "id_gsd-ud-dev.conllu")
PATH_GSD_TEST  = os.path.join(BASE_PATH, "UD_Indonesian_GSD", "id_gsd-ud-test.conllu")

### 2. Exploration Raw Data

In [4]:
# ============================================
# BACA DATA GSD
# ============================================
with open(PATH_GSD_TRAIN, "r", encoding="utf-8") as f:
    data_gsd_train = conllu.parse(f.read())
with open(PATH_GSD_DEV, "r", encoding="utf-8") as f:
    data_gsd_dev = conllu.parse(f.read())
with open(PATH_GSD_TEST, "r", encoding="utf-8") as f:
    data_gsd_test = conllu.parse(f.read())

data_gsd_all = data_gsd_train + data_gsd_dev + data_gsd_test

# ============================================
# AMBIL SAMPLE RAW DATA DARI GSD
# ============================================
raw_data = []

for kalimat in data_gsd_all:
    teks = kalimat.metadata.get('text', '')
    sent_id = kalimat.metadata.get('sent_id', '')
    
    for token in kalimat:
        if not isinstance(token['id'], int):
            continue
            
        raw_data.append({
            'sent_id'  : sent_id,
            'teks'     : teks,
            'token_id' : token['id'],
            'token'    : token['form'],
            'lemma'    : token['lemma'],
            'upos'     : token['upos'],
            'xpos'     : token['xpos'],
            'feats'    : str(token['feats']) if token['feats'] else '-',
            'head'     : token['head'],
            'deprel'   : token['deprel'],
            'misc'     : str(token['misc']) if token['misc'] else '-',
        })

df_raw = pd.DataFrame(raw_data)

# ============================================
# TAMPILKAN STATISTIK LABEL
# ============================================
print("=" * 60)
print("INFORMASI RAW DATA UD-INDO-GSD")
print("=" * 60)

print(f"\n{'Jumlah kalimat (all)':<30}: {len(data_gsd_all)}")
print(f"{'Jumlah kalimat (train)':<30}: {len(data_gsd_train)}")
print(f"{'Jumlah kalimat (dev)':<30}: {len(data_gsd_dev)}")
print(f"{'Jumlah kalimat (test)':<30}: {len(data_gsd_test)}")

# Hitung total token
total_token = sum(
    1 for kalimat in data_gsd_all
    for token in kalimat
    if isinstance(token['id'], int)
)
total_token_no_punct = sum(
    1 for kalimat in data_gsd_all
    for token in kalimat
    if isinstance(token['id'], int) and token['upos'] != 'PUNCT'
)
print(f"\n{'Total token':<30}: {total_token}")
print(f"{'Total token (no punct)':<30}: {total_token_no_punct}")

# ============================================
# TABEL 1: FIELD YANG TERSEDIA
# ============================================
print("\n\n=== TABEL: FIELD YANG TERSEDIA ===")
fields_info = {
    'sent_id' : 'ID kalimat',
    'token'   : 'Bentuk token asli',
    'lemma'   : 'Bentuk dasar token',
    'upos'    : 'Universal POS tag',
    'xpos'    : 'Language-specific POS tag',
    'feats'   : 'Morphological features',
    'head'    : 'ID token kepala (dependency)',
    'deprel'  : 'Dependency relation label',
    'misc'    : 'Informasi tambahan (MorphInd)',
}

df_fields = pd.DataFrame([
    {'FIELD': k, 'KETERANGAN': v}
    for k, v in fields_info.items()
])
display(df_fields)

# ============================================
# TABEL 2: DISTRIBUSI UPOS
# ============================================
print("\n\n=== TABEL: DISTRIBUSI UPOS ===")
upos_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if isinstance(token['id'], int) and token['upos'] != 'PUNCT':
            upos_data.append(token['upos'])

upos_counter = Counter(upos_data)
total = sum(upos_counter.values())

df_upos = pd.DataFrame([
    {
        'UPOS'       : upos,
        'JUMLAH'     : count,
        'KETERANGAN' : {
            'NOUN' : 'Kata benda',
            'VERB' : 'Kata kerja',
            'ADJ'  : 'Kata sifat',
            'ADV'  : 'Kata keterangan',
            'PROPN': 'Kata benda proper',
            'DET'  : 'Determiner',
            'ADP'  : 'Adposisi',
            'PRON' : 'Kata ganti',
            'CCONJ': 'Konjungsi koordinatif',
            'SCONJ': 'Konjungsi subordinatif',
            'NUM'  : 'Numeralia',
            'AUX'  : 'Kata bantu',
            'PART' : 'Partikel',
            'INTJ' : 'Interjeksi',
            'SYM'  : 'Simbol',
            'X'    : 'Lainnya',
        }.get(upos, '-')
    }
    for upos, count in upos_counter.most_common()
])
display(df_upos)

# ============================================
# TABEL 3: DISTRIBUSI XPOS
# ============================================

# KETERANGAN XPOS (Language-Specific POS Tag - MorphInd/IDT tagset)
XPOS_KETERANGAN = {
    # ── NOMINA (N) ──────────────────────────────────────────────────
    'NSD'     : 'Nomina dasar (common noun)',
    'NSD--'   : 'Nomina dasar tanpa fitur morfologi',
    'NSD-SG'  : 'Nomina dasar bentuk tunggal',
    'NSD-PL'  : 'Nomina dasar bentuk jamak (reduplikasi)',
    'NSD-SG-' : 'Nomina dasar tunggal tanpa fitur tambahan',
    'NSD-PL-' : 'Nomina dasar jamak tanpa fitur tambahan',
    'NSP'     : 'Nomina proper (nama diri)',
    'NSP--'   : 'Nomina proper tanpa fitur morfologi',
    'NSP-SG'  : 'Nomina proper bentuk tunggal',
    'NSP-PL'  : 'Nomina proper bentuk jamak',

    # ── VERBA (V) ────────────────────────────────────────────────────
    'VS--'    : 'Verba dasar tanpa fitur (bare verb)',
    'VSA'     : 'Verba aktif (berafiks me-)',
    'VSA-'    : 'Verba aktif tanpa fitur tambahan',
    'VSA--'   : 'Verba aktif tanpa fitur morfologi',
    'VSAP'    : 'Verba aktif-pasif (dapat aktif dan pasif)',
    'VSP'     : 'Verba pasif (berafiks di-/ter-)',
    'VSP-'    : 'Verba pasif tanpa fitur tambahan',
    'VSP--'   : 'Verba pasif tanpa fitur morfologi',
    'VSB'     : 'Verba benefaktif (berafiks memper-/diper-)',
    'VSB-'    : 'Verba benefaktif tanpa fitur tambahan',
    'VSD'     : 'Verba dasar (stative/intransitive)',
    'VSD-'    : 'Verba dasar tanpa fitur tambahan',

    # ── ADJEKTIVA (A) ────────────────────────────────────────────────
    'AS--'    : 'Adjektiva tanpa fitur morfologi',
    'ASP'     : 'Adjektiva positif (derajat biasa)',
    'ASP-'    : 'Adjektiva positif tanpa fitur tambahan',
    'ASC'     : 'Adjektiva komparatif (lebih ...)',
    'ASC-'    : 'Adjektiva komparatif tanpa fitur tambahan',
    'ASS'     : 'Adjektiva superlatif (paling ...)',
    'ASS-'    : 'Adjektiva superlatif tanpa fitur tambahan',

    # ── ADVERBIA (D) ─────────────────────────────────────────────────
    'D--'     : 'Adverbia umum',
    'D---'    : 'Adverbia tanpa fitur morfologi',
    'D-F'     : 'Adverbia frekuensi (selalu, sering, jarang)',
    'D-T'     : 'Adverbia temporal (kemarin, besok, sudah)',
    'D-M'     : 'Adverbia modalitas (mungkin, tentu, pasti)',
    'D-N'     : 'Adverbia negasi (tidak, bukan, belum, jangan)',

    # ── PREPOSISI / ADPOSISI (R) ─────────────────────────────────────
    'R--'     : 'Preposisi umum (di, ke, dari, dengan)',
    'R---'    : 'Preposisi tanpa fitur morfologi',

    # ── KONJUNGSI KOORDINATIF (CC) ───────────────────────────────────
    'CC--'    : 'Konjungsi koordinatif (dan, atau, tetapi)',
    'CC---'   : 'Konjungsi koordinatif tanpa fitur morfologi',

    # ── KONJUNGSI SUBORDINATIF (SC) ──────────────────────────────────
    'SC--'    : 'Konjungsi subordinatif (bahwa, karena, jika, meski)',
    'SC---'   : 'Konjungsi subordinatif tanpa fitur morfologi',

    # ── PRONOMINA (P) ────────────────────────────────────────────────
    'PS1'     : 'Pronomina persona pertama (saya, aku, kami, kita)',
    'PS1-SG'  : 'Pronomina persona pertama tunggal (saya, aku)',
    'PS1-PL'  : 'Pronomina persona pertama jamak (kami, kita)',
    'PS2'     : 'Pronomina persona kedua (kamu, Anda, kalian)',
    'PS2-SG'  : 'Pronomina persona kedua tunggal (kamu, Anda)',
    'PS2-PL'  : 'Pronomina persona kedua jamak (kalian)',
    'PS3'     : 'Pronomina persona ketiga (dia, ia, mereka)',
    'PS3-SG'  : 'Pronomina persona ketiga tunggal (dia, ia)',
    'PS3-PL'  : 'Pronomina persona ketiga jamak (mereka)',
    'PD--'    : 'Pronomina demonstratif (ini, itu, sini, situ)',
    'PI--'    : 'Pronomina interogatif (apa, siapa, mana)',
    'PR--'    : 'Pronomina relatif (yang)',
    'PIN-'    : 'Pronomina indefinit (sesuatu, seseorang, masing-masing)',
    'PN--'    : 'Pronomina negatif (tidak seorang pun, tidak ada)',

    # ── NUMERALIA (Q) ────────────────────────────────────────────────
    'Q--'     : 'Numeralia kardinal (satu, dua, 10, 100)',
    'Q---'    : 'Numeralia kardinal tanpa fitur morfologi',
    'QO-'     : 'Numeralia ordinal (pertama, kedua, ke-3)',
    'QO--'    : 'Numeralia ordinal tanpa fitur tambahan',
    'QF-'     : 'Numeralia fraksional (setengah, seperempat)',
    'QC-'     : 'Numeralia kolektif (kedua-duanya, bertiga)',

    # ── DETERMINER (T) ───────────────────────────────────────────────
    'T--'     : 'Determiner/penentu (para, sang, si, kaum)',
    'T---'    : 'Determiner tanpa fitur morfologi',

    # ── KATA BANTU / AUXILIARI (O) ───────────────────────────────────
    'O--'     : 'Kata bantu (akan, sedang, telah, sudah, dapat, harus)',
    'O---'    : 'Kata bantu tanpa fitur morfologi',

    # ── PARTIKEL (G) ─────────────────────────────────────────────────
    'G--'     : 'Partikel penegas (-lah, -kah, -pun, toh)',
    'G---'    : 'Partikel tanpa fitur morfologi',

    # ── KATA SERU / INTERJEKSI (I) ───────────────────────────────────
    'I--'     : 'Interjeksi / kata seru (wah, aduh, hei, ya)',

    # ── KATA ASING (F) ───────────────────────────────────────────────
    'F--'     : 'Kata asing (foreign word, belum diserap)',

    # ── TANDA BACA / SIMBOL (Z) ──────────────────────────────────────
    'Z--'     : 'Tanda baca (titik, koma, tanda tanya)',
    'Z---'    : 'Tanda baca tanpa fitur morfologi',

    # ── TIDAK TERKLASIFIKASI (X) ─────────────────────────────────────
    'X--'     : 'Kategori tidak terklasifikasi / kata asing tak dikenal',
    'X---'    : 'Tidak terklasifikasi tanpa fitur morfologi',
    '_'       : 'Tidak ada xpos (token tanpa anotasi xpos)',
}

print("\n\n=== TABEL: DISTRIBUSI XPOS ===")

xpos_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if isinstance(token['id'], int) and token['upos'] != 'PUNCT':
            xpos_data.append(token['xpos'])

xpos_counter = Counter(xpos_data)
total_xpos   = sum(xpos_counter.values())

df_xpos = pd.DataFrame([
    {
        'XPOS'       : xpos,
        'JUMLAH'     : count,
        'KETERANGAN' : XPOS_KETERANGAN.get(xpos, '-'),
    }
    for xpos, count in xpos_counter.most_common()
])
display(df_xpos)

# ============================================
# TABEL 4: DISTRIBUSI DEPREL
# ============================================

# KETERANGAN DEPREL
DEPREL_KETERANGAN = {
    # Core arguments
    'root'       : 'Kata utama kalimat',
    'nsubj'      : 'Subjek nominal',
    'nsubj:pass' : 'Subjek nominal kalimat pasif',
    'obj'        : 'Objek langsung',
    'iobj'       : 'Objek tidak langsung',
    'csubj'      : 'Subjek klausal',
    'csubj:pass' : 'Subjek klausal kalimat pasif',
    'ccomp'      : 'Komplemen klausal',
    'xcomp'      : 'Komplemen klausal terbuka',

    # Nominal dependents
    'nmod'       : 'Modifier nominal',
    'nmod:poss'  : 'Modifier posesif',
    'appos'      : 'Aposisi',
    'nummod'     : 'Modifier numerik',
    'amod'       : 'Modifier adjektival',
    'det'        : 'Determiner',
    'clf'        : 'Classifier',
    'case'       : 'Preposisi/postposisi',

    # Verb dependents
    'obl'        : 'Keterangan oblique',
    'obl:agent'  : 'Keterangan agen (pasif)',
    'advmod'     : 'Modifier adverbial',
    'aux'        : 'Kata bantu',
    'aux:pass'   : 'Kata bantu pasif',
    'cop'        : 'Kopula',
    'compound'   : 'Kata majemuk',
    'compound:plur' : 'Reduplikasi (kata ulang jamak)',
    'flat'       : 'Ekspresi flat (nama)',
    'flat:name'  : 'Nama diri',
    'fixed'      : 'Ekspresi tetap',

    # Coordination
    'conj'       : 'Konjungsi koordinatif',
    'cc'         : 'Konjungsi koordinatif (kata)',
    'cc:preconj' : 'Konjungsi pre-koordinatif',

    # Clausal dependents
    'acl'        : 'Klausa adjektival',
    'acl:relcl'  : 'Klausa relatif',
    'advcl'      : 'Klausa adverbial',
    'mark'       : 'Penanda klausa (bahwa/yang)',
    'parataxis'  : 'Hubungan parataktis',
    'list'       : 'Hubungan list',
    'orphan'     : 'Orphan (ellipsis)',
    'reparandum' : 'Koreksi ujaran',
    'discourse'  : 'Penanda wacana',

    # Special
    'punct'      : 'Tanda baca',
    'dep'        : 'Dependensi tidak terspesifikasi',
    'goeswith'   : 'Bagian dari kata yang sama',
    'vocative'   : 'Vokativa',
    'expl'       : 'Ekspletif',
    'dislocated' : 'Dislokasi',
}


print("=== TABEL DISTRIBUSI DEPREL ===")

deprel_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if isinstance(token['id'], int) and token['upos'] != 'PUNCT':
            deprel_data.append(token['deprel'])

deprel_counter = Counter(deprel_data)
total_deprel   = sum(deprel_counter.values())

df_deprel = pd.DataFrame([
    {
        'DEPREL'     : deprel,
        'JUMLAH'     : count,
        'KETERANGAN' : DEPREL_KETERANGAN.get(deprel, '-')
    }
    for deprel, count in deprel_counter.most_common()
])
display(df_deprel)


# ============================================
# TABEL 5: DISTRIBUSI FEATS
# ============================================

# KETERANGAN MORPHOLOGICAL FEATURES

FEATS_KETERANGAN = {
    # Voice
    'Voice=Act'         : 'Verba aktif (me-)',
    'Voice=Pass'        : 'Verba pasif (di-)',

    # Number
    'Number=Sing'       : 'Bentuk tunggal',
    'Number=Plur'       : 'Bentuk jamak',

    # PronType
    'PronType=Rel'      : 'Pronomina relatif (yang)',
    'PronType=Dem'      : 'Pronomina demonstratif (ini/itu)',
    'PronType=Ind'      : 'Pronomina indefinit (sebuah)',
    'PronType=Prs'      : 'Pronomina persona (saya/dia)',
    'PronType=Int'      : 'Pronomina interogatif (apa/siapa)',
    'PronType=Tot'      : 'Pronomina total (semua)',
    'PronType=Neg'      : 'Pronomina negatif (tidak ada)',

    # Degree
    'Degree=Pos'        : 'Adjektiva positif',
    'Degree=Cmp'        : 'Adjektiva komparatif (lebih)',
    'Degree=Sup'        : 'Adjektiva superlatif (paling)',

    # NumType
    'NumType=Card'      : 'Numeralia kardinal (satu, dua)',
    'NumType=Ord'       : 'Numeralia ordinal (pertama, kedua)',
    'NumType=Frac'      : 'Numeralia pecahan (setengah)',
    'NumType=Mult'      : 'Numeralia multiplikatif (dua kali)',
    'NumType=Dist'      : 'Numeralia distributif',
    'NumType=Sets'      : 'Numeralia set',

    # Person
    'Person=1'          : 'Persona pertama (saya/kami)',
    'Person=2'          : 'Persona kedua (kamu/Anda)',
    'Person=3'          : 'Persona ketiga (dia/mereka)',

    # Polarity
    'Polarity=Neg'      : 'Negasi (tidak/bukan/belum)',
    'Polarity=Pos'      : 'Positif (afirmatif)',

    # Possessor
    'Number[psor]=Sing' : 'Pemilik tunggal (-nya/-ku/-mu)',
    'Number[psor]=Plur' : 'Pemilik jamak',
    'Person[psor]=1'    : 'Pemilik persona pertama (-ku)',
    'Person[psor]=2'    : 'Pemilik persona kedua (-mu)',
    'Person[psor]=3'    : 'Pemilik persona ketiga (-nya)',

    # Polite
    'Polite=Form'       : 'Bentuk formal/hormat (Anda)',
    'Polite=Infm'       : 'Bentuk informal (kamu)',

    # Clusivity
    'Clusivity=Ex'      : 'Eksklusif (kami — tidak termasuk lawan bicara)',
    'Clusivity=In'      : 'Inklusif (kita — termasuk lawan bicara)',

    # Gender
    'Gender=Masc'       : 'Maskulin',
    'Gender=Fem'        : 'Feminin',

    # Mood
    'Mood=Imp'          : 'Modus imperatif (perintah)',
    'Mood=Ind'          : 'Modus indikatif (pernyataan)',
    'Mood=Sub'          : 'Modus subjunktif',

    # Aspect
    'Aspect=Perf'       : 'Aspek perfektif (sudah)',
    'Aspect=Imp'        : 'Aspek imperfektif (sedang)',
    'Aspect=Iter'       : 'Aspek iteratif (berulang)',
}

print("\n\n=== TABEL DISTRIBUSI MORPHOLOGICAL FEATURES ===")

feats_data = []
for kalimat in data_gsd_all:
    for token in kalimat:
        if not isinstance(token['id'], int):
            continue
        if token['upos'] == 'PUNCT':
            continue
        if token['feats']:
            for key, val in token['feats'].items():
                feats_data.append(f"{key}={val}")

feats_counter = Counter(feats_data)
total_feats   = sum(feats_counter.values())

df_feats = pd.DataFrame([
    {
        'FEATURE'    : feat,
        'JUMLAH'     : count,
        'KETERANGAN' : FEATS_KETERANGAN.get(feat, '-')
    }
    for feat, count in feats_counter.most_common()
])
display(df_feats)


# ============================================
# TABEL 6: SAMPLE RAW DATA LENGKAP
# ============================================
print("\n\n=== TABEL: SAMPLE RAW DATA (20 baris pertama) ===")
display(df_raw[['sent_id', 'teks', 'token', 'lemma', 'upos', 'xpos',
                'feats', 'head', 'deprel', 'misc']].head(20))

INFORMASI RAW DATA UD-INDO-GSD

Jumlah kalimat (all)          : 5593
Jumlah kalimat (train)        : 4477
Jumlah kalimat (dev)          : 559
Jumlah kalimat (test)         : 557

Total token                   : 121923
Total token (no punct)        : 103695


=== TABEL: FIELD YANG TERSEDIA ===


,FIELD,KETERANGAN
0,sent_id,ID kalimat
1,token,Bentuk token asli
2,lemma,Bentuk dasar token
3,upos,Universal POS tag
4,xpos,Language-specific POS tag
5,feats,Morphological features
6,head,ID token kepala (dependency)
7,deprel,Dependency relation label
8,misc,Informasi tambahan (MorphInd)




=== TABEL: DISTRIBUSI UPOS ===


,UPOS,JUMLAH,KETERANGAN
0,NOUN,27000,Kata benda
1,PROPN,22790,Kata benda proper
2,VERB,12202,Kata kerja
3,ADP,12019,Adposisi
4,PRON,4764,Kata ganti
5,ADV,4760,Kata keterangan
6,ADJ,4528,Kata sifat
7,NUM,4383,Numeralia
8,DET,4012,Determiner
9,CCONJ,3659,Konjungsi koordinatif




=== TABEL: DISTRIBUSI XPOS ===


...

,DEPREL,JUMLAH,KETERANGAN
0,case,11897,Preposisi/postposisi
1,flat,11402,Ekspresi flat (nama)
2,compound,7428,Kata majemuk
3,nsubj,7125,Subjek nominal
4,obl,6346,Keterangan oblique
5,obj,5794,Objek langsung
6,root,5592,Kata utama kalimat
7,advmod,5288,Modifier adverbial
8,conj,4806,Konjungsi koordinatif
9,amod,4566,Modifier adjektival




=== TABEL DISTRIBUSI MORPHOLOGICAL FEATURES ===


,FEATURE,JUMLAH,KETERANGAN
0,Number=Sing,49801,Bentuk tunggal
1,Voice=Act,9352,Verba aktif (me-)
2,Degree=Pos,6001,Adjektiva positif
3,NumType=Card,4714,"Numeralia kardinal (satu, dua)"
4,Voice=Pass,3527,Verba pasif (di-)
5,PronType=Rel,3110,Pronomina relatif (yang)
6,PronType=Dem,2048,Pronomina demonstratif (ini/itu)
7,Number[psor]=Sing,1877,Pemilik tunggal (-nya/-ku/-mu)
8,Person[psor]=3,1785,Pemilik persona ketiga (-nya)
9,PronType=Ind,1138,Pronomina indefinit (sebuah)




=== TABEL: SAMPLE RAW DATA (20 baris pertama) ===


,sent_id,teks,token,lemma,upos,xpos,feats,head,deprel,misc
0,train-s1,Sembungan adalah sebuah desa yang terletak di ...,Sembungan,sembungan,PROPN,X--,-,4,nsubj,{'MorphInd': '^sembungan<x>_X--$'}
1,train-s1,Sembungan adalah sebuah desa yang terletak di ...,adalah,adalah,AUX,O--,-,4,cop,{'MorphInd': '^adalah<o>_O--$'}
2,train-s1,Sembungan adalah sebuah desa yang terletak di ...,sebuah,sebuah,DET,B--,{'PronType': 'Ind'},4,det,{'MorphInd': '^sebuah<b>_B--$'}
3,train-s1,Sembungan adalah sebuah desa yang terletak di ...,desa,desa,NOUN,NSD,{'Number': 'Sing'},0,root,{'MorphInd': '^desa<n>_NSD$'}
4,train-s1,Sembungan adalah sebuah desa yang terletak di ...,yang,yang,PRON,S--,{'PronType': 'Rel'},6,nsubj:pass,{'MorphInd': '^yang<s>_S--$'}
5,train-s1,Sembungan adalah sebuah desa yang terletak di ...,terletak,terletak,VERB,VSP,"{'Number': 'Sing', 'Voice': 'Pass'}",4,acl,{'MorphInd': '^ter+letak<n>_VSP$'}
6,train-s1,Sembungan adalah sebuah desa yang terletak di ...,di,di,ADP,R--,-,8,case,{'MorphInd': '^di<r>_R--$'}
7,train-s1,Sembungan adalah sebuah desa yang terletak di ...,kecamatan,kecamatan,NOUN,NSD,{'Number': 'Sing'},6,obl,{'MorphInd': '^ke+camat<n>+an_NSD$'}
8,train-s1,Sembungan adalah sebuah desa yang terletak di ...,Kejajar,kejajar,PROPN,X--,-,8,flat,"{'SpaceAfter': 'No', 'MorphInd': '^kejajar<x>_..."
9,train-s1,Sembungan adalah sebuah desa yang terletak di ...,",",",",PUNCT,Z--,-,8,punct,"{'MorphInd': '^,<z>_Z--$'}"


## B. Explore Task Dataset - IndoLEM POS Tagging (RAW)

### 1. Set Path

In [5]:
BASE_PATH_POS = r"D:\UNIVERSITASSEBELASMARET\Semester8\File TA\Progress\Progress TA\SourceCode\TugasAkhir_MuhamadFaqihZacky\data\raw\indolem-main\pos_tagging\data"

PATHS_POS_TRAIN = [os.path.join(BASE_PATH_POS, f"train.0{i}.tsv") for i in range(1, 6)]
PATHS_POS_DEV   = [os.path.join(BASE_PATH_POS, f"dev.0{i}.tsv") for i in range(1, 6)]
PATHS_POS_TEST  = [os.path.join(BASE_PATH_POS, f"test.0{i}.tsv") for i in range(1, 6)]

### 2. Exploration Raw Data

In [7]:
# ============================================
# BACA DATA INDOLEM POS TAGGING (5-FOLD)
# ============================================
def baca_tsv_pos(filepaths):
    sentences = []
    for filepath in filepaths:
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                content = f.read().strip()
                if content:
                    sentences.extend(content.split('\n\n'))
        except FileNotFoundError:
            continue
    return sentences

# Menggunakan list variable dari cell 1 (PATHS_POS_TRAIN, dll)
data_pos_train = baca_tsv_pos(PATHS_POS_TRAIN)
data_pos_dev = baca_tsv_pos(PATHS_POS_DEV)
data_pos_test = baca_tsv_pos(PATHS_POS_TEST)

data_pos_all = data_pos_train + data_pos_dev + data_pos_test

# ============================================
# AMBIL SAMPLE RAW DATA DARI INDOLEM POS
# ============================================
raw_data = []
sent_id_counter = 1

for kalimat in data_pos_all:
    lines = kalimat.split('\n')
    token_id = 1
    
    # Gabungkan token untuk membentuk teks utuh
    teks = " ".join([line.split('\t')[0] for line in lines if len(line.split('\t')) == 2])
    
    for line in lines:
        parts = line.split('\t')
        if len(parts) == 2:
            raw_data.append({
                'sent_id'  : sent_id_counter,
                'teks'     : teks,
                'token_id' : token_id,
                'token'    : parts[0],
                'pos_tag'  : parts[1]
            })
            token_id += 1
    sent_id_counter += 1

df_raw = pd.DataFrame(raw_data)

# ============================================
# TAMPILKAN STATISTIK LABEL
# ============================================
print("=" * 60)
print("INFORMASI RAW DATA INDOLEM POS TAGGING")
print("=" * 60)

print(f"\n{'Jumlah kalimat (all)':<30}: {len(data_pos_all)}")
print(f"{'Jumlah kalimat (train)':<30}: {len(data_pos_train)}")
print(f"{'Jumlah kalimat (dev)':<30}: {len(data_pos_dev)}")
print(f"{'Jumlah kalimat (test)':<30}: {len(data_pos_test)}")

# HITUNG TOTAL TOKEN PER SPLIT + ALL
def hitung_token(data, include_punct=True):
    return sum(
        1 for kalimat in data
        for token in kalimat
        if isinstance(token['id'], int)
        and (include_punct or token['upos'] != 'PUNCT')
    )

splits = {
    'train' : data_gsd_train,
    'dev'   : data_gsd_dev,
    'test'  : data_gsd_test,
    'all'   : data_gsd_all,
}

print(f"\n{'Split':<10} {'Token (incl. PUNCT)':>22} {'Token (excl. PUNCT)':>22}")
print("-" * 56)
for nama, data in splits.items():
    total        = hitung_token(data, include_punct=True)
    total_no_punc = hitung_token(data, include_punct=False)
    print(f"{nama:<10} {total:>22,} {total_no_punc:>22,}")

# ============================================
# TABEL 1: FIELD YANG TERSEDIA
# ============================================
print("\n\n=== TABEL 1: FIELD YANG TERSEDIA ===")
fields_info = {
    'sent_id' : 'ID kalimat (dibuat otomatis)',
    'teks'    : 'Teks utuh hasil gabungan token',
    'token_id': 'Urutan token dalam kalimat',
    'token'   : 'Bentuk token asli (kata)',
    'pos_tag' : 'Part-of-Speech tag spesifik Bahasa Indonesia',
}

df_fields = pd.DataFrame([
    {'FIELD': k, 'KETERANGAN': v}
    for k, v in fields_info.items()
])
display(df_fields)

# ============================================
# TABEL 2: DISTRIBUSI POS TAG (INDOLEM)
# ============================================
# KETERANGAN TAG POS INDONESIA (Standard tagset)
POS_KETERANGAN = {
    'NN'  : ('Nomina', 'kata benda umum, menyatakan objek atau konsep'),
    'NNP' : ('Nomina Propria', 'kata benda nama diri seperti orang, tempat, institusi'),
    'NND' : ('Nomina Penunjuk Arah', 'menyatakan lokasi relatif seperti atas, bawah, dalam'),
    
    'VB'  : ('Verba', 'kata kerja yang menyatakan aksi, proses, atau keadaan'),
    'MD'  : ('Modal', 'kata bantu yang menyatakan aspek, waktu, atau kemungkinan seperti akan, sudah, dapat'),
    
    'JJ'  : ('Adjektiva', 'kata sifat yang menyatakan keadaan atau kualitas'),
    'RB'  : ('Adverbia', 'kata keterangan yang memodifikasi verba, adjektiva, atau kalimat'),
    
    'PRP' : ('Pronomina Persona', 'kata ganti orang seperti saya, dia, mereka'),
    'PR'  : ('Pronomina Demonstrativa', 'kata ganti penunjuk seperti ini, itu'),
    
    'IN'  : ('Preposisi', 'kata depan yang menunjukkan hubungan seperti di, ke, dari'),
    'CC'  : ('Konjungsi Koordinatif', 'penghubung setara seperti dan, atau'),
    'SC'  : ('Konjungsi Subordinatif', 'penghubung tidak setara seperti karena, bahwa, jika'),
    
    'CD'  : ('Numeralia', 'kata bilangan seperti satu, dua, 10'),
    'OD'  : ('Numeralia Ordinal', 'bilangan tingkat seperti pertama, kedua'),
    
    'NEG' : ('Negasi', 'kata penyangkalan seperti tidak, bukan, belum'),
    'DT'  : ('Determiner', 'kata penentu seperti para, sang, si'),
    'WH'  : ('Kata Tanya', 'digunakan dalam kalimat interogatif seperti apa, siapa, kapan'),
    
    'RP'  : ('Partikel', 'kata tugas yang memberi penekanan seperti -lah, -pun'),
    'UH'  : ('Interjeksi', 'kata seru yang mengekspresikan emosi seperti wah, aduh'),
    
    'FW'  : ('Foreign Word', 'kata asing yang belum diserap ke dalam bahasa Indonesia'),
    'SYM' : ('Simbol', 'lambang non-alfabet seperti %, $, #'),
    'Z'   : ('Tanda Baca', 'punctuation seperti titik, koma, tanda tanya'),
    
    'X'   : ('Unknown', 'token yang tidak dapat diklasifikasikan atau kesalahan penulisan')
}

print("\n\n=== TABEL 2: DISTRIBUSI POS TAG ===")

pos_counter = Counter(df_raw[df_raw['pos_tag'].notnull()]['pos_tag'])

df_pos = pd.DataFrame([
    {
        'POS TAG'    : tag,
        'JUMLAH'     : count,
        'KATEGORI'   : POS_KETERANGAN.get(tag, ('-', '-'))[0],
        'KETERANGAN' : POS_KETERANGAN.get(tag, ('-', '-'))[1]
    }
    for tag, count in pos_counter.most_common()
])
pd.set_option('display.max_colwidth', None)
display(df_pos)


# ============================================
# TABEL 3: SAMPLE RAW DATA LENGKAP
# ============================================
print("\n\n=== TABEL 3: SAMPLE RAW DATA (20 baris pertama) ===")
display(df_raw[['sent_id', 'teks', 'token_id', 'token', 'pos_tag']].head(20))

INFORMASI RAW DATA INDOLEM POS TAGGING

Jumlah kalimat (all)          : 50150
Jumlah kalimat (train)        : 36110
Jumlah kalimat (dev)          : 4010
Jumlah kalimat (test)         : 10030

Split         Token (incl. PUNCT)    Token (excl. PUNCT)
--------------------------------------------------------
train                      97,531                 82,963
dev                        12,612                 10,676
test                       11,780                 10,056
all                       121,923                103,695


=== TABEL 1: FIELD YANG TERSEDIA ===


,FIELD,KETERANGAN
0,sent_id,ID kalimat (dibuat otomatis)
1,teks,Teks utuh hasil gabungan token
2,token_id,Urutan token dalam kalimat
3,token,Bentuk token asli (kata)
4,pos_tag,Part-of-Speech tag spesifik Bahasa Indonesia




=== TABEL 2: DISTRIBUSI POS TAG ===


,POS TAG,JUMLAH,KATEGORI,KETERANGAN
0,NN,329975,Nomina,"kata benda umum, menyatakan objek atau konsep"
1,NNP,174045,Nomina Propria,"kata benda nama diri seperti orang, tempat, institusi"
2,VB,160620,Verba,"kata kerja yang menyatakan aksi, proses, atau keadaan"
3,Z,131735,Tanda Baca,"punctuation seperti titik, koma, tanda tanya"
4,IN,106570,Preposisi,"kata depan yang menunjukkan hubungan seperti di, ke, dari"
5,CD,89800,Numeralia,"kata bilangan seperti satu, dua, 10"
6,SC,66055,Konjungsi Subordinatif,"penghubung tidak setara seperti karena, bahwa, jika"
7,JJ,49315,Adjektiva,kata sifat yang menyatakan keadaan atau kualitas
8,PRP,37915,Pronomina Persona,"kata ganti orang seperti saya, dia, mereka"
9,CC,37220,Konjungsi Koordinatif,"penghubung setara seperti dan, atau"




=== TABEL 3: SAMPLE RAW DATA (20 baris pertama) ===


,sent_id,teks,token_id,token,pos_tag
0,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",1,Pemerintah,NN
1,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",2,bahkan,RB
2,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",3,telah,MD
3,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",4,mencanangkan,VB
4,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",5,dana,NN
5,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",6,untuk,SC
6,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",7,memicu,VB
7,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",8,sektor,NN
8,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",9,usaha,NN
9,1,"Pemerintah bahkan telah mencanangkan dana untuk memicu sektor usaha kecil dan menengah UKM tumbuh lebih baik , karena sektor ini cukup kuat dalam krisis keuangan pada tahun 1997 lalu , kata -nya .",10,kecil,JJ


# Data Annotator